# Iterative methods
Lecture 7 companion. Run cells in order; all data are generated here. Requires NumPy, SciPy and Matplotlib. Saved plots are ordinary images, so the notebook works in Jupyter and PyCharm without embedded lecture PDFs.

We solve nonlinear equations and fit a nonlinear model using local linear approximations. The hand-written iterations expose the mathematics; they are not general-purpose replacements for library solvers.

References: [Boyd and Vandenberghe, §§18.1–18.4](https://web.stanford.edu/~boyd/vmls/vmls.pdf); SciPy [`root`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.root.html), [`least_squares`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.least_squares.html).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import root, least_squares

# Figures are collected here so the same plots can be used in the slides.
figures = {}

## Overview
We begin with nonlinear equations and ODE fixed points, then develop Newton's method and its use of Jacobians and linear solves. For measured data we discuss transformations, nonlinear least squares and Gauss–Newton. We close by comparing the linear and nonlinear solvers used in these examples.

Throughout, we define the function to pass to a solver, supply an initial guess where needed, interpret the returned results and check convergence.

### Checkpoint: ODE fixed points
Consider $\dot x=\cos x-x$. Which initial state makes the solution remain constant? At a fixed point, $\dot x=0$, so we must solve $\cos x^*=x^*$.

**Answer on the following plot:** draw $y=\cos x$ and $y=x$. Their intersection has equal function values, approximately $(0.739,0.739)$, so the state $x^*\approx0.739$ makes the ODE's rate zero. The graph gives an initial estimate; the `root` example later refines it without integrating an entire trajectory.

In [ ]:
# A fixed point occurs where the two function values agree.
ode_states = np.linspace(-0.5, 1.5, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(ode_states, np.cos(ode_states), label=r"$y=\cos x$")
ax.plot(ode_states, ode_states, "--", label=r"$y=x$")
ax.set(xlabel="x", ylabel="y", xlim=(-0.5, 1.5))
ax.legend()
figures["ode-fixed-point"] = fig
plt.show()

## 1. A scalar equation
Find the positive solution of $f(x)=x^2-2=0$. Newton replaces the curve near the current guess by its tangent. The tangent reaches zero at
$$x_{k+1}=x_k-\frac{f(x_k)}{f'(x_k)}.$$
Start at $x_0=2$. The derivative is $f'(x)=2x$.

### From correction to the next point
The Newton correction is a signed movement, $\Delta x_k=-f(x_k)/f'(x_k)$, not a replacement location. A full step gives $x_{k+1}=x_k+\Delta x_k$. A damped variant multiplies the correction by a fraction $0<\alpha_k\leq1$; all the hand-written loops here use full steps, $\alpha_k=1$.

Starting at 2 for $f(x)=x^2-2$: $f(2)=2$, $f'(2)=4$, so the correction is $-0.5$ and the new point is $2-0.5=1.5$. Recompute at 1.5: $f(1.5)=0.25$ and $f'(1.5)=3$. The next correction is $-1/12$, giving $1.5-1/12=17/12$, approximately 1.416667. A half first step would instead give $2+0.5(-0.5)=1.75$.

In [ ]:
def scalar_f(x):
    return x*x - 2.0

def scalar_df(x):
    return 2.0*x

x = 2.0
scalar_history = [x]
for k in range(5):
    correction = -scalar_f(x)/scalar_df(x)
    x = x + correction
    scalar_history.append(x)
    print("Iteration", k + 1, "estimate", x, "f(x)", scalar_f(x))

In [ ]:
# Show how one tangent produces the next guess.
xx = np.linspace(0.5, 2.3, 200)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xx, scalar_f(xx), label="f(x) = x squared - 2")
ax.plot(xx, scalar_f(2.0) + scalar_df(2.0)*(xx - 2.0),
        "--", label="Tangent at x = 2")
ax.axhline(0.0, color="black")
ax.plot([2.0], [scalar_f(2.0)], "o", label="Current point")
ax.plot([1.5], [0.0], "o", label="Next guess: 1.5")
ax.set(xlabel="x", ylabel="Function / tangent value")
ax.legend()
figures["newton-tangent"] = fig
plt.show()

In [ ]:
# Compare estimates with the exact positive root.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(6), scalar_history, "o-", label="Newton estimates")
ax.axhline(np.sqrt(2.0), color="orange", linestyle="--", label="Square root of 2")
ax.set(xlabel="Iteration", ylabel="Estimate")
ax.legend()
figures["scalar-convergence"] = fig
plt.show()

### Checkpoint: Newton
For $f(x)=x^2-2$, can we take a Newton step starting at 0? Does failure mean the equation has no root?

**Answer:** The derivative is zero, so the basic step would divide by zero. The roots still exist; this initial guess is unsuitable.

### A simple stopping rule
For $f(x)=0$, stop when $|f(x_k)|<\varepsilon$, and set a maximum number of iterations in case this condition is never met. For our example, use $|x_k^2-2|<10^{-6}$.

The actual solution error is $e_k=|x_k-x^*|$. Since the exact root is usually unknown, we check the function value instead. A small function value does not directly guarantee the same small error in the root; its meaning depends on the equation's scale.

`ax.semilogy` uses a logarithmic vertical axis: moving from $10^{-2}$ to $10^{-4}$ means a hundred-fold reduction.

In [ ]:
# Check the equation value after each Newton update.
stop_guesses = np.array(scalar_history[:5])
stop_values = np.abs(scalar_f(stop_guesses))
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(5), stop_values, "o-", label=r"$|f(x_k)|=|x_k^2-2|$")
ax.axhline(1e-6, color="black", linestyle="--", label="Stop below tolerance: 1e-6")
ax.set(xlabel="Iteration k", ylabel="Absolute function value (logarithmic scale)")
ax.set_xticks(range(5))
ax.legend()
figures["stopping-history"] = fig
plt.show()

### Small updates can mean stagnation
The next plot is an intentionally constructed **illustration**, not a Newton run. We seek the root 1 of $f(x)=x-1$, start at 2, and make successively smaller movements $-0.1,-0.01,-0.001,\ldots$. The sequence approaches about 1.889 instead of 1. The updates tend to zero although the equation is not satisfied.

In [ ]:
stalled_steps = 10.0**(-np.arange(1, 8))
stalled_guesses = 2.0 - np.cumsum(stalled_steps)
stalled_values = np.abs(stalled_guesses - 1.0)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(1, 8), stalled_steps, "s-", label=r"$d_k=|x_k-x_{k-1}|$")
ax.semilogy(range(1, 8), stalled_values, "o-", label=r"$v_k=|f(x_k)|$")
ax.axhline(1e-6, color="black", linestyle="--", label="Illustrative tolerance: 1e-6")
ax.set(xlabel="Iteration k", ylabel="Size (logarithmic scale)")
ax.set_xticks(range(1, 8))
ax.legend()
figures["stalled-history"] = fig
plt.show()

### Checkpoint: which converges faster?
Two methods start at 0.8 and seek the root of $\cos x-x=0$. We use the high-accuracy reference $x^*\approx0.7390851332151607$ to plot the error $|x_k-x^*|$. Unlike a function-value check, this error needs a known reference and is not generally available during an actual solve. Which method reaches error below $10^{-6}$ first? Does fewer iterations guarantee less time?

In [ ]:
rate_reference = 0.7390851332151607
method_a = [0.8]
method_b = [0.8]
for k in range(3):
    # A: use the equation in the form x = cos(x).
    old_a = method_a[-1]
    method_a.append(np.cos(old_a))
    # B: a full Newton update for cos(x) - x = 0.
    old_b = method_b[-1]
    change_b = -(np.cos(old_b)-old_b)/(-np.sin(old_b)-1.0)
    method_b.append(old_b + change_b)
error_a = np.abs(np.array(method_a)-rate_reference)
error_b = np.abs(np.array(method_b)-rate_reference)
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(4), error_a, "o-", label="Method A")
ax.semilogy(range(4), error_b, "s-", label="Method B")
ax.axhline(1e-6, color="black", linestyle="--", label="Target error: 1e-6")
ax.set(xlabel="Iteration k", ylabel=r"$e_k=|x_k-x^*|$ (logarithmic scale)")
ax.set_xticks(range(4))
ax.legend()
figures["rate-question"] = fig
plt.show()

### Answer: local convergence rate
B is Newton's method and reaches the target after two updates. A repeatedly sets the new guess to the cosine of the old guess; near this root it removes roughly a fixed fraction of the remaining error (linear convergence). Newton converges quadratically near a simple root under appropriate smoothness assumptions: the next error is roughly proportional to the square of the current error.

“Linear convergence” describes the error reduction, not whether the original equation is linear. Fewer updates do not guarantee less runtime: evaluate the cost of each update too. Newton needs derivatives; for systems it also needs linear solves.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogy(range(4), error_a, "o-", label=r"A: set new x to $\cos(x)$")
ax.semilogy(range(4), error_b, "s-", label="B: Newton's method")
ax.axhline(1e-6, color="black", linestyle="--", label="Target error: 1e-6")
ax.set(xlabel="Iteration k", ylabel=r"$e_k=|x_k-x^*|$ (logarithmic scale)")
ax.set_xticks(range(4))
ax.legend()
figures["rate-answer"] = fig
plt.show()

The rate examples refer to the actual solution error $e_k=|x_k-x^*|$, not to the function value or the change between guesses. The linear example $e_{k+1}=0.1e_k$ gives errors $0.1,0.01,0.001,0.0001$. The quadratic example $e_{k+1}=e_k^2$ gives $0.1,0.01,0.0001,0.00000001$. These are illustrative local-rate sequences. Newton is often locally quadratic near a simple smooth root, but its starting point still matters; eventually rounding can prevent further improvement.

### Stopping and failure
For an equation written as $f(x)=0$, evaluate its left-hand side at the current guess. In our example that value is $x^2-2$. A root makes it zero. This function value is different from the unknown error $x-x^*$ in the guessed root.

A small update alone does not prove that the equation is satisfied. A useful solver limits the number of steps and checks whether the absolute function value is below a chosen tolerance. Real problems also need appropriate scaling and units.

Newton needs a usable derivative and a suitable initial guess. For $g(x)=x^3-2x+2$, starting at 0 gives the cycle $0\to1\to0$. Near a simple root, smooth functions can give rapid convergence, but it is not guaranteed from every starting point.

In [ ]:
def scalar_newton(f, derivative, x0, equation_tol=1e-10, max_steps=20):
    x = float(x0)
    history = [x]
    for k in range(max_steps):
        if abs(f(x)) <= equation_tol:
            return x, history, "Equation tolerance reached"
        slope = derivative(x)
        if abs(slope) < 1e-14:
            return x, history, "Derivative too small for this example"
        correction = -f(x)/slope
        x = x + correction
        history.append(x)
    return x, history, "Iteration limit reached; check the function value"

scalar_answer, checked_history, scalar_message = scalar_newton(scalar_f, scalar_df, 2.0)
print(scalar_answer, scalar_message)

def cycle_f(x):
    return x**3 - 2*x + 2

def cycle_df(x):
    return 3*x*x - 2

cycle_answer, cycle_history, cycle_message = scalar_newton(cycle_f, cycle_df, 0.0, max_steps=6)
print("Cycle:", cycle_history)
print(cycle_message, "Function value:", cycle_f(cycle_answer))

In [ ]:
xx = np.linspace(-2.0, 1.4, 250)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(xx, cycle_f(xx), label="g(x) = x cubed - 2x + 2")
ax.plot(xx, 2.0 - 2.0*xx, "--", label="Tangent at 0 gives 1")
ax.plot(xx, xx, ":", label="Tangent at 1 gives 0")
ax.axhline(0.0, color="black")
ax.plot([0, 1], [0, 0], "o", label="Alternating guesses")
ax.set(xlabel="x", ylabel="Function / tangent value")
ax.legend()
figures["newton-cycle"] = fig
plt.show()

## 2. Two nonlinear equations
Find the intersection of a circle and parabola:
$$F(z)=\begin{bmatrix}u^2+v^2-1\\v-u^2\end{bmatrix}=0,
\qquad z=\begin{bmatrix}u\\v\end{bmatrix}.$$
A root makes **both** components zero. We use the positive-$u$ intersection.

The Jacobian contains first derivatives, with **one row per equation and one column per unknown**:
$$J_F(z)=\begin{bmatrix}2u&2v\\-2u&1\end{bmatrix}.$$
Given $z_k$, approximate $F(z_k+\Delta z)\approx F(z_k)+J_F(z_k)\Delta z$. Set this approximation to zero and solve for the correction. This is not the ODE stability test: the same derivative matrix is now used to construct a numerical update.

`equations(z)` evaluates the function value at one state guess; it does not solve for a root. The solver repeatedly calls it at different guesses. At `[0.8, 0.8]`, the function value is `[0.28, 0.16]` and the first correction is approximately `[-0.00576923, -0.16923077]`. `np.linalg.solve(A, b)` returns the vector solving the square nonsingular system. `np.linalg.norm(equations(z))` is the Euclidean function value size: the square root of the sum of squared entries.

In [ ]:
def equations(z):
    u, v = z
    return np.array([u*u + v*v - 1.0, v - u*u])

def jacobian(z):
    u, v = z
    return np.array([[2*u, 2*v], [-2*u, 1.0]])

# Each row is one successive guess for (u, v).
z = np.array([0.8, 0.8])
system_history = [z.copy()]
for k in range(5):
    correction = np.linalg.solve(jacobian(z), -equations(z))
    z = z + correction
    system_history.append(z.copy())
    print("Iteration", k + 1, "state", z, "equation-value norm", np.linalg.norm(equations(z)))
system_history = np.array(system_history)

In [ ]:
# The zero contours are the original equations.
u_values = np.linspace(-1.15, 1.15, 250)
v_values = np.linspace(-0.1, 1.15, 220)
U, V = np.meshgrid(u_values, v_values)
fig, ax = plt.subplots(figsize=(8, 4))
ax.contour(U, V, U*U + V*V - 1, levels=[0], colors=["tab:blue"])
ax.contour(U, V, V - U*U, levels=[0], colors=["tab:orange"])
ax.plot([], [], color="tab:blue", label="Circle")
ax.plot([], [], color="tab:orange", label="Parabola")
ax.plot(system_history[:, 0], system_history[:, 1], "o-", color="black", label="Newton guesses")
ax.set(xlabel="u", ylabel="v")
ax.set_aspect("equal")
ax.legend(loc="lower left")
figures["system-intersection"] = fig
plt.show()

In [ ]:
# Replace both equations by lines around the same current guess.
z0 = np.array([0.8, 0.8])
first_correction = np.linalg.solve(jacobian(z0), -equations(z0))
z1 = z0 + first_correction
F0 = equations(z0)
J0 = jacobian(z0)
linear_first = F0[0] + J0[0, 0]*(U-z0[0]) + J0[0, 1]*(V-z0[1])
linear_second = F0[1] + J0[1, 0]*(U-z0[0]) + J0[1, 1]*(V-z0[1])
fig, ax = plt.subplots(figsize=(8, 4))
ax.contour(U, V, U*U+V*V-1, levels=[0], colors=["tab:blue"])
ax.contour(U, V, V-U*U, levels=[0], colors=["tab:orange"])
ax.contour(U, V, linear_first, levels=[0], colors=["tab:blue"], linestyles="dashed")
ax.contour(U, V, linear_second, levels=[0], colors=["tab:orange"], linestyles="dashed")
ax.plot([], [], "--", color="black", label="Linearised equations")
ax.plot([z0[0], z1[0]], [z0[1], z1[1]], "o-", color="black", label="First correction")
ax.set(xlabel="u", ylabel="v", xlim=(0.3, 1.05), ylim=(0.3, 1.05))
ax.set_aspect("equal")
ax.legend(loc="lower left")
figures["system-linearisation"] = fig
plt.show()

### Checkpoint: the correction
For our two-variable system, what shapes do the Jacobian and correction have? Why add the correction? Must the nonlinear equation vector become zero immediately?

**Answers:** Shapes `(2, 2)` and `(2,)`. The correction is a change from the current state. It makes the local linear approximation zero, not necessarily the original nonlinear equations, so we evaluate and repeat.

### Let SciPy manage the iterations
Construct the equation function by subtracting right-hand side from left-hand side in **each** equation. For $G(z)=c$, return $G(z)-c$. In our example, return the vector `[u*u + v*v - 1.0, v - u*u]`. Both entries must be zero at a solution. Return all the expressions together, rather than their sum or Boolean equality tests. The function evaluates a guess; `root` searches for a guess that zeros the vector, and can fail to converge.

`root(fun, x0, jac=..., method="hybr", tol=...)` finds a zero of a vector function. `fun(z)` returns an array of equation values, and `jac(z)` returns the Jacobian. Here there are two unknowns and two equations. `x0` is a guess, not the data or the answer.

The default `hybr` is Powell's hybrid method, not exactly our bare Newton loop. It incorporates safeguards. A tolerance controls termination; it is not a guaranteed bound on the actual solution error. When `jac` is omitted, the default method estimates derivatives numerically.

Pass the callable `equations`, not an already-evaluated array `equations(z)`. For `hybr`, `tol` sets a relative-step termination test. A returned small step does not replace checking the actual equation function value.

In [ ]:
root_result = root(equations, [0.8, 0.8], jac=jacobian,
                   method="hybr", tol=1e-10)
print("Solution:", root_result.x)
print("Equation values:", root_result.fun)
print("Success:", root_result.success)
print("Message:", root_result.message)
print("Function evaluations:", root_result.nfev)

Try `[-0.8, 0.8]` as the initial guess. A different root is not necessarily an error: these equations have two intersections. Always check the returned function value, solver status and the physical meaning of a proposed solution.

In [ ]:
other_root = root(equations, [-0.8, 0.8], jac=jacobian)
print("Other intersection:", other_root.x)
print("Equation-value norm:", np.linalg.norm(other_root.fun))

### Return to the ODE example
Use the same `root` interface to find the equilibrium of the scalar ODE introduced at the start. Even with one unknown, `root` supplies a length-1 state array, so its equation function should return one function value. No time argument is needed because this autonomous equilibrium problem only depends on the state.

In [ ]:
def ode_equilibrium_function(state):
    return np.cos(state) - state

ode_equilibrium = root(ode_equilibrium_function, [0.8])
print("Fixed point:", ode_equilibrium.x)
print("Rate at the fixed point:", ode_equilibrium.fun)
print("Success:", ode_equilibrium.success)

## 3. Nonlinear least squares
### Physical context: cooling
A warm object loses heat to a room at approximately constant temperature. Newton's law of cooling models the rate as $\dot T=-b(T-T_{\rm room})$, with $b>0$. Its solution is $T(t)=T_{\rm room}+a e^{-bt}$. Thus the excess temperature $y(t)=T(t)-T_{\rm room}$ follows $a e^{-bt}$.

Here $a$ is the initial temperature above room temperature and $b$ controls the cooling rate (larger means faster). If time is measured in minutes, $b$ has units of inverse minutes. Our simulated signal uses illustrative units and has the known room-temperature baseline already subtracted. The model is an approximation: the object's temperature is treated as uniform and heat exchange is assumed proportional to the temperature difference.

[Background: OpenStax, Newton's law of cooling](https://openstax.org/books/calculus-volume-2/pages/2-8-exponential-growth-and-decay).

A cooling signal is modelled as $m(t;a,b)=a e^{-bt}$, where $a$ is its initial magnitude and $b$ its decay rate. We measure a noisy signal above a known baseline (already subtracted). Units are illustrative. Real data may need an additional baseline parameter.

The measurements cannot generally all lie on one curve. Define $r_i(a,b)=m(t_i;a,b)-y_i$ and minimise $L(a,b)=\frac12\sum_i r_i^2$. The residual function must return **all residuals**, not their sum or their squared norm.

Unlike polynomial regression, the unknown $b$ occurs inside the exponential, so the model is nonlinear **in its parameters**. We fit the original measurements directly, without a logarithmic transformation.

Data-generation functions: `np.linspace(0, 5, 18)` returns 18 evenly spaced times including both endpoints. `np.exp` acts on every array entry. `rng.normal(0, 0.06, t.size)` draws 18 Gaussian noise values with mean zero and standard deviation 0.06; 0.06 is not a maximum error bound. The seed makes this simulated dataset repeatable. The true parameters used to generate it are known to us, but the fitting procedure receives only the measurements and an initial parameter guess.

### Checkpoint: estimating coefficients by transformation
Given measurements $(t_i,T_i)$ and known $T_{\rm room}$, can a transformation turn fitting $T_i\approx T_{\rm room}+ae^{-bt_i}$ into linear regression? How do we recover $a,b$? What if all differences are negative? Do transformed and original-data fits minimise the same errors? What if room temperature is unknown?

### Answers
Let $y_i=T_i-T_{\rm room}$. If the nonzero differences all share a sign, fit
$$z_i=\log|y_i|\approx\beta_0+\beta_1t_i.$$
The design matrix has columns of ones and times. Recover $b=-\beta_1$ and $a=e^{\beta_0}$ for positive differences, or $a=-e^{\beta_0}$ for negative differences. The latter is equivalent to fitting $\log(-y_i)$. The sign cases are different:
- For $y=[4,2,1]$, use $\log y$ and recover positive $a$.
- For $y=[-4,-2,-1]$, use $\log(-y)$, which is the log of $[4,2,1]$, and recover negative $a$.
- If a difference is zero, its logarithm is undefined, so this direct transformation cannot use all observations.
- For mixed nonzero signs, such as $[2,-1,0.5]$, $\log|y|$ is defined, but it discards the signs. Because $e^{-bt}>0$, the signed exponential $ae^{-bt}$ always has the same sign as $a$. No single sign of $a$ restores both positive and negative observations after the absolute-value transformation. Fit the original signed residuals instead.

Log regression minimises squared errors in transformed data, so it generally gives different coefficients from minimising squared temperature errors. Our simulated example has additive noise in the original measurement units, and we minimise those errors directly. With unknown room temperature we cannot form a fixed transformed dataset by subtracting it; instead we can estimate room temperature, $a$ and $b$ together with nonlinear least squares.

The nonnegative amplitude bound later in the notebook is specific to the positive-difference dataset; it is not a restriction on the general model.

In [ ]:
# Create the dataset before any fitting.
rng = np.random.RandomState(587)  # Fixed seed for repeatable lecture figures.
t = np.linspace(0.0, 5.0, 18)
y = 2.5*np.exp(-0.7*t) + rng.normal(0.0, 0.06, t.size)

def model(t, parameters):
    a, b = parameters
    return a*np.exp(-b*t)

def residuals(parameters, t, y):
    return model(t, parameters) - y

initial_parameters = np.array([2.0, 0.4])
print("First three times:", t[:3])
print("First three measurements:", y[:3])
print("Initial residual vector shape:", residuals(initial_parameters, t, y).shape)

In [ ]:
# Draw each residual as the vertical gap from data to model.
t_fine = np.linspace(0.0, 5.0, 250)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y, "o", label="Measurements")
ax.plot(t_fine, model(t_fine, initial_parameters), label="Initial curve")
ax.vlines(t, y, model(t, initial_parameters), color="gray", label="Residual gaps")
ax.set(xlabel="Time", ylabel="Signal above baseline")
ax.legend()
figures["decay-initial"] = fig
plt.show()

### Checkpoint: fitting
Is $a+bt^2$ or $a e^{-bt}$ linear in its unknown parameters? With 18 measurements and 2 parameters, should the residual function return 2 numbers, 18 signed differences, or one sum of squares?

**Answers:** $a+bt^2$ is linear in the parameters even though it is curved in time. The exponential is nonlinear in the decay parameter. Return 18 signed differences; SciPy constructs the objective from them.

### Gauss–Newton
For a correction $\Delta\theta$, linearise the residual vector:
$$r(\theta_k+\Delta\theta)\approx r(\theta_k)+J_r(\theta_k)\Delta\theta.$$
Choose the correction by solving
$$\min_{\Delta\theta}\|J_r(\theta_k)\Delta\theta+r(\theta_k)\|_2.$$
Here $J_r$ has 18 rows and 2 columns; the correction has 2 entries. This is the linear least-squares problem already studied, with matrix $J_r$ and right-hand side $-r$.

For this model, row $i$ of the Jacobian is $[e^{-bt_i},-a t_i e^{-bt_i}]$. A full Gauss–Newton step can fail or increase the objective; the following undamped loop is a demonstration for this initial guess. Practical solvers control the step.

`np.column_stack` places two derivative arrays side by side, creating the `(18, 2)` Jacobian. `np.linalg.lstsq(J, -r, rcond=None)` returns a tuple; its first entry is the length-2 correction. Other entries contain residual sums, numerical rank and singular values. `rcond=None` lets NumPy choose its numerical-rank threshold. SVD knowledge is not needed to use this call here.

In [ ]:
def residual_jacobian(parameters, t, y):
    a, b = parameters
    exponential = np.exp(-b*t)
    return np.column_stack((exponential, -a*t*exponential))

parameters = initial_parameters.copy()
fit_history = [parameters.copy()]
fit_costs = [0.5*np.sum(residuals(parameters, t, y)**2)]
for k in range(6):
    r = residuals(parameters, t, y)
    J = residual_jacobian(parameters, t, y)
    correction = np.linalg.lstsq(J, -r, rcond=None)[0]
    parameters = parameters + correction
    fit_history.append(parameters.copy())
    fit_costs.append(0.5*np.sum(residuals(parameters, t, y)**2))
    print("Iteration", k+1, "parameters", parameters, "cost", fit_costs[-1])
fit_history = np.array(fit_history)

In [ ]:
# Plot curves from a few successive parameter estimates.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y, "o", color="black", label="Measurements")
for k in [0, 1, 2, 6]:
    ax.plot(t_fine, model(t_fine, fit_history[k]), label="Iteration " + str(k))
ax.set(xlabel="Time", ylabel="Signal above baseline")
ax.legend()
figures["decay-iterations"] = fig
plt.show()

### Use a practical nonlinear least-squares solver
`least_squares(fun, x0, jac="2-point", bounds=(-np.inf, np.inf), method="trf", args=())` is a selected-argument summary, **not the full signature**. Default `trf` is a trust-region reflective method. Here we explicitly supply the analytic Jacobian and nonnegative bounds. The nonnegative amplitude bound is chosen for this warmer-object dataset. For an initially colder object, allow a negative amplitude; the cooling rate remains positive.

`args=(t, y)` passes both data arrays to `residuals(parameters, t, y)` and `residual_jacobian(parameters, t, y)`. A tuple with one argument needs a trailing comma, for example `args=(t,)`.

The solver chooses its steps; this call is not an execution of exactly the hand-written full-step Gauss–Newton loop.

The first list in `bounds` contains the lower limits for `[a, b]`; the second gives their upper limits. `np.inf` removes an upper limit. Trust-region step control means the solver limits how far it trusts a local approximation and adjusts the step when the model predicts poorly. Set `max_nfev=200`, for example, to limit residual-function evaluations.

In [ ]:
fit = least_squares(residuals, initial_parameters,
                    jac=residual_jacobian, args=(t, y),
                    bounds=([0.0, 0.0], [np.inf, np.inf]),
                    method="trf")
print("Fitted a and b:", fit.x)
print("Cost = half the sum of squared residuals:", fit.cost)
print("Residual vector shape:", fit.fun.shape)
print("Success:", fit.success)
print("Message:", fit.message)
print("Function evaluations:", fit.nfev)
print("First-order optimality measure:", fit.optimality)

In [ ]:
# Compare the fitted curve with the measurements.
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(t, y, "o", label="Measurements")
ax.plot(t_fine, model(t_fine, fit.x), label="Fitted curve")
ax.set(xlabel="Time", ylabel="Signal above baseline")
ax.legend()
figures["decay-fit"] = fig
plt.show()

In [ ]:
# Colour represents cost; the path represents Gauss-Newton iterates.
a_grid = np.linspace(1.7, 3.0, 100)
b_grid = np.linspace(0.3, 1.05, 100)
AA, BB = np.meshgrid(a_grid, b_grid)
C = np.zeros_like(AA)
for time, measurement in zip(t, y):
    C = C + 0.5*(AA*np.exp(-BB*time)-measurement)**2
fig, ax = plt.subplots(figsize=(8, 4))
contours = ax.contour(AA, BB, C, levels=[0.05, 0.1, 0.3, 0.8, 1.5, 3.0])
ax.clabel(contours)
ax.plot(fit_history[:, 0], fit_history[:, 1], "o-", label="Gauss-Newton updates")
ax.plot([fit.x[0]], [fit.x[1]], "x", markersize=10, label="SciPy fit")
ax.set(xlabel="Initial magnitude a", ylabel="Decay rate b")
ax.legend()
figures["parameter-path"] = fig
plt.show()

### Interpreting the result
`fit.x` contains the fitted parameters; `fit.fun` contains $r_i$; `fit.cost` is $\tfrac12\sum r_i^2$ with the default linear loss. The default termination controls `ftol`, `xtol` and `gtol` are each $10^{-8}$: small cost change, small parameter step, or a first-order optimality criterion can stop the solver. They do **not** bound the error relative to the true physical parameters. `nfev` counts function evaluations, not iterations; accounting for finite-difference evaluations is method/version dependent.

`success` says a termination criterion was met, not that a global minimum or correct physical model was found. Try several sensible initial guesses; inspect residuals and constraints. Nonlinear least squares is generally nonconvex. Also compare with omitting `jac=...`: SciPy estimates derivatives numerically by default.

For the unconstrained squared-error objective the local slope is described by the gradient `J.T @ r`. With bounds, SciPy uses a bound-aware first-order measure reported as `optimality`. A small measure indicates local stationarity, not necessarily small residuals or a global minimum.

In [ ]:
numerical_jacobian_fit = least_squares(residuals, initial_parameters,
                                       args=(t, y), bounds=(0.0, np.inf))
print("Fit with numerical derivatives:", numerical_jacobian_fit.x)
print("Agreement:", np.allclose(fit.x, numerical_jacobian_fit.x, rtol=1e-5, atol=1e-7))

for guess in [[1.0, 0.2], [4.0, 1.0]]:
    trial = least_squares(residuals, guess, args=(t, y), bounds=(0.0, np.inf))
    print("Start", guess, "fit", trial.x, "cost", trial.cost)

## Solver guide
- `np.linalg.solve(A, b)`: a square nonsingular linear system.
- `np.linalg.lstsq(A, b, rcond=None)[0]`: linear least squares; in Gauss–Newton this gives a parameter correction.
- `scipy.optimize.root(F, z0)`: nonlinear equations, with one returned equation value per unknown for our method.
- `scipy.optimize.least_squares(r, theta0)`: minimise squared measurement residuals for a nonlinear model.

For Newton and Gauss–Newton: evaluate at the current guess, solve the local linear problem, add the correction, then check again.

## Try it
1. Change Newton's starting point. Can a different start lead to a different root or prevent progress?
2. Change the cooling data's noise level or initial parameter guess. Compare the fitted curves and residuals.
3. Explain why `root`, `least_squares`, `solve` and `lstsq` solve different mathematical problems.

Next lecture: SVD and conditioning explain why a computed answer can be sensitive to the data.